# Notebook 23 — CReST vs. feature-based Ridge: domain-relative prediction

**Objective.** Test whether *CReST general + domain-specific learned representations* provide additional information about **hidden-target domain performance conditional on official PISA 2022 Creative Thinking proficiency**, compared with a straightforward feature-based Ridge baseline.

**Uses existing artifacts; no Transformer retraining.** The original CReST hidden-target latent archives and plausible-value table from **Notebook 09**, and the feature definition, A/B split and source-only handling from **Notebook 15**. This compares the **original, published CReST**, not the exploratory v2 pilot; no v2 cross-system hidden-domain latent archives are available.

**Matched evaluation.** Seven outer folds (45 training / 9 validation / 9 test systems), both A→B/B→A directions, same source and target items, same eligible student records, same 10 plausible values, validation-only `StandardScaler` + `Ridge(alpha=1)`, and system-equal Spearman correlations. Both general-only and general+domain versions are fitted separately for each method. All 63 education systems contribute to the paired analyses when eligibility allows.

**Important methodological qualifications.**

- The *general CReST* branch is 64 dimensions and *general+domain CReST* is 128 dimensions. The feature Ridge has 36 general and 12 additional domain-specific source-only features (48 total). The difference measures the utility of the full representation configurations, **not** an architecture-only effect.
- PISA plausible values are used exclusively to construct **the same conditional targets** as Notebook 09. They are **never model inputs**. Test-system outcomes participate only in Notebook 09's previously specified within-system residual **target definition** and scoring, **not in supervised Ridge fitting**. This is an analysis of conditional prediction profiles, **not a deployable prospective predictor**.
- A/B is a *scored-item-level* split, not a task-level split. Neither method is being assessed on entirely unseen tasks.
- This baseline was designed *after examining earlier CReST and feature-baseline results*. Treat resulting comparisons as **exploratory / post hoc**; the 63 systems here are not a newly independent confirmatory test set.

**Execution.** Place this notebook in the same project root as the other notebooks. Run **all** code cells in order. It saves CSV tables, protocol information, optional reproducibility comparisons and a single ZIP under `results/crest_vs_ridge_domain_relative/`. It does not write into any existing `processed/`, `checkpoints/`, or CReST result folders.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cell 1 — paths, unchanged protocol, imports
from pathlib import Path
import os, json, hashlib, zipfile
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

BASE_DIR = Path(os.environ.get('CREST_PROJECT_DIR', Path.cwd())).expanduser().resolve()
HANDOFF_DIR = BASE_DIR / 'processed' / 'crest_v5_handoff'
FINAL_TEST_ROOT = BASE_DIR / 'processed' / 'crest_final_test_sets'
PROFILE_DIR = BASE_DIR / 'results' / 'profile_beyond_overall_proficiency'
LATENT_DIR = PROFILE_DIR / 'hidden_target_domain_latents'
OUTPUT_DIR = BASE_DIR / 'results' / 'crest_vs_ridge_domain_relative'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ITEM_SET_A = np.array([0,1,5,6,7,8,9,10,12,15,17,20,24,27,28,31], dtype=np.int64)
ITEM_SET_B = np.array([2,3,4,11,13,14,16,18,19,21,22,23,25,26,29,30], dtype=np.int64)
DIRECTIONS = {'A_to_B': (ITEM_SET_A, ITEM_SET_B), 'B_to_A': (ITEM_SET_B, ITEM_SET_A)}
DOMAIN_ORDER = ['Written', 'Visual', 'Social', 'Scientific']
PV_COLUMNS = [f'PV{i}CRTH_NC' for i in range(1,11)]
ALPHA = 1.0
MIN_SYSTEM_RESID_N = 30
MIN_SYSTEM_METRIC_N = 50
BOOTSTRAP_REPS = 10_000
BOOTSTRAP_SEED = 20261016
assert np.array_equal(np.sort(np.r_[ITEM_SET_A,ITEM_SET_B]),np.arange(32))
print('Project root:', BASE_DIR)
print('Output directory:', OUTPUT_DIR)

Project root: D:\Kubra (Umit blg)\Kubra (papers)\PISA-CReST\Dataset and codes
Output directory: D:\Kubra (Umit blg)\Kubra (papers)\PISA-CReST\Dataset and codes\results\crest_vs_ridge_domain_relative


In [3]:
# Cell 2 — fail early if Notebook 01/09 artifacts are unavailable
required = {
    'cohort metadata': HANDOFF_DIR / 'primary_meta.pkl',
    'handoff config': HANDOFF_DIR / 'handoff_config.json',
    'PV table from Notebook 09': PROFILE_DIR / 'primary_cohort_with_pisa_ct_pvs.pkl',
    'CReST system results from Notebook 09': PROFILE_DIR / 'crest_profile_beyond_pisa_ct_system.csv',
}
for k in ['credit_tokens','behaviour_tokens','performance_mask','interaction_mask',
          'process_tensor','item_domain_ids','item_ideation_ids']:
    required[k] = HANDOFF_DIR / f'{k}.npy'
for fold in range(7):
    required[f'held-out test indices fold {fold}'] = FINAL_TEST_ROOT / f'fold_{fold}' / 'test_indices.npy'
    for split in ('validation','test'):
        for direction in DIRECTIONS:
            required[f'latent fold {fold} {split} {direction}'] = LATENT_DIR / f'fold_{fold}_{split}_{direction}.npz'
missing = [(name,str(path)) for name,path in required.items() if not path.is_file()]
if missing:
    print('Missing required files (first 20):')
    for name,path in missing[:20]: print('  ',name,'->',path)
    raise FileNotFoundError(f'{len(missing)} required file(s) missing. Run the source Notebook 09 latent/PV/results cells and 01/06 preparation. See list above.')
print(f'Input audit PASS: {len(required)} required files found; no model training needed.')

Input audit PASS: 46 required files found; no model training needed.


In [4]:
# Cell 3 — load source arrays and official plausible values from Notebook 09
with open(HANDOFF_DIR/'handoff_config.json',encoding='utf-8') as f:
    config = json.load(f)
primary_meta = pd.read_pickle(HANDOFF_DIR/'primary_meta.pkl').reset_index(drop=True)
pv_meta = pd.read_pickle(PROFILE_DIR/'primary_cohort_with_pisa_ct_pvs.pkl').reset_index(drop=True)
credit = np.load(HANDOFF_DIR/'credit_tokens.npy',mmap_mode='r')
behaviour = np.load(HANDOFF_DIR/'behaviour_tokens.npy',mmap_mode='r')
performance_mask = np.load(HANDOFF_DIR/'performance_mask.npy',mmap_mode='r')
interaction_mask = np.load(HANDOFF_DIR/'interaction_mask.npy',mmap_mode='r')
process = np.load(HANDOFF_DIR/'process_tensor.npy',mmap_mode='r')
domain_ids = np.load(HANDOFF_DIR/'item_domain_ids.npy',mmap_mode='r')
ideation_ids = np.load(HANDOFF_DIR/'item_ideation_ids.npy',mmap_mode='r')
N=len(primary_meta)
assert N==142_564==len(pv_meta)==int(config['n_students'])
assert primary_meta['CNT'].nunique()==63
assert all(primary_meta['CNT'].astype(str).to_numpy() == pv_meta['CNT'].astype(str).to_numpy()), 'PV metadata row order mismatch'
assert 'handoff_row_index' in pv_meta.columns
assert np.array_equal(pv_meta['handoff_row_index'].to_numpy(dtype=np.int64),np.arange(N)), 'PV table row indices do not match handoff'
if 'CNTSTUID' in primary_meta and 'CNTSTUID' in pv_meta:
    assert np.array_equal(primary_meta['CNTSTUID'].to_numpy(), pv_meta['CNTSTUID'].to_numpy()), 'PV IDs do not align'
assert credit.shape == behaviour.shape == performance_mask.shape == interaction_mask.shape == (N,32)
assert process.shape==(N,32,3)
assert domain_ids.shape==(32,) and ideation_ids.shape==(32,)
assert np.array_equal(np.sort(np.unique(domain_ids)),np.arange(4))
assert np.array_equal(np.sort(np.unique(ideation_ids)),np.arange(3))
assert config.get('process_order') == ['GDI','GCI','EII']
assert np.isin(np.unique(credit),[0,1,2,3]).all()
assert np.isin(np.unique(behaviour),[0,1,2,3]).all()
assert all(col in pv_meta.columns for col in PV_COLUMNS)
fold_values=primary_meta['country_fold'].to_numpy(dtype=np.int64)
country_values=primary_meta['CNT'].astype(str).to_numpy()
pv_matrix=pv_meta[PV_COLUMNS].to_numpy(dtype=np.float32)
assert set(fold_values)==set(range(7))
assert primary_meta.groupby('CNT')['country_fold'].nunique().eq(1).all()
print(f'Data audit PASS: {N:,} pupils, 63 systems, 32 items, 10 plausible values.')

Data audit PASS: 142,564 pupils, 63 systems, 32 items, 10 plausible values.


## Source-only feature construction

The general summary uses Notebook 15's **3 ideation groups × 12 features = 36 columns**: mean normalized scored credit; proportions of three response-behaviour states; mean available log-transformed process channels; and five observation-presence flags. The new *domain* summary computes **the same 12 statistics only from source items in that task domain**, irrespective of ideation-process label. The general and general+domain models share the same eligible pupils and conditional outcome.

Do **not** include target-item credit, target behavioural/process information, target-item availability, PV scores, or country/booklet IDs as predictors. `StandardScaler` is fitted only on the validation-system students used to fit each Ridge probe.

In [5]:
# Cell 4 — faithful Notebook 15 36-feature general baseline plus 12-feature domain summary

def masked_mean(values, valid):
    valid = np.asarray(valid,dtype=bool)
    values = np.asarray(values,dtype=np.float64)
    total = np.where(valid,values,0.0).sum(axis=1)
    count = valid.sum(axis=1)
    mean = np.divide(total,count,out=np.zeros(len(total),dtype=np.float64),where=count>0)
    return mean,(count>0).astype(np.float64)


def summary_block(indices, source_items):
    """12 source-only summary values, identical statistics and ordering to Notebook 15."""
    rows = np.asarray(indices,dtype=np.int64)
    items = np.asarray(source_items,dtype=np.int64)
    assert len(items)>0 and np.isin(items,np.arange(32)).all()
    ct = np.asarray(credit[np.ix_(rows,items)])
    bt = np.asarray(behaviour[np.ix_(rows,items)])
    mc = np.asarray(performance_mask[np.ix_(rows,items)],dtype=bool) & np.isin(ct,[1,2,3])
    mi = np.asarray(interaction_mask[np.ix_(rows,items)],dtype=bool) & np.isin(bt,[1,2,3])
    px = np.asarray(process[rows][:,items,:],dtype=np.float64)
    cols=[]
    mean_credit, credit_present=masked_mean((ct.astype(float)-1.0)/2.0,mc)
    cols.append(mean_credit)
    count_b=mi.sum(axis=1)
    for state in (1,2,3):
        numer=((bt==state)&mi).sum(axis=1)
        cols.append(np.divide(numer,count_b,out=np.zeros(len(rows),dtype=float),where=count_b>0))
    process_availability=[]
    for j in range(3):
        valid=np.isfinite(px[:,:,j]) & mi
        avg,present=masked_mean(px[:,:,j],valid)
        cols.append(avg)
        process_availability.append(present)
    cols.extend([credit_present,(count_b>0).astype(float)])
    cols.extend(process_availability)
    x=np.column_stack(cols).astype(np.float32)
    assert x.shape==(len(rows),12) and np.isfinite(x).all()
    return x


def general_summary(indices, source_items):
    all_blocks=[]
    for ideation in range(3):
        these=np.asarray(source_items,dtype=np.int64)
        these=these[np.asarray(ideation_ids[these])==ideation]
        if len(these)==0: raise ValueError(f'No source items for ideation {ideation}')
        all_blocks.append(summary_block(indices,these))
    x=np.concatenate(all_blocks,axis=1)
    assert x.shape==(len(indices),36)
    return x


def domain_summary(indices, source_items, domain_id):
    these=np.asarray(source_items,dtype=np.int64)
    these=these[np.asarray(domain_ids[these])==int(domain_id)]
    if len(these)==0: raise ValueError(f'No source items in domain {domain_id}')
    return summary_block(indices,these)


def hidden_domain_target(indices, target_items):
    rows=np.asarray(indices,dtype=np.int64)
    items=np.asarray(target_items,dtype=np.int64)
    ct=np.asarray(credit[np.ix_(rows,items)])
    pm=np.asarray(performance_mask[np.ix_(rows,items)],dtype=bool)
    assert np.all(~pm | np.isin(ct,[1,2,3])), 'Scorable mask includes unexpected credit token'
    values=np.where(pm,(ct.astype(np.float32)-1.0)/2.0,0.0)
    counts=pm.sum(axis=1)
    y=np.divide(values.sum(axis=1),np.maximum(counts,1)).astype(np.float32)
    y[counts==0]=np.nan
    return y,counts


def source_domain_observed(indices, source_items):
    rows=np.asarray(indices,dtype=np.int64)
    items=np.asarray(source_items,dtype=np.int64)
    return np.asarray(interaction_mask[np.ix_(rows,items)],dtype=bool).any(axis=1)

print('Source-only 36+12 feature builders ready.')

Source-only 36+12 feature builders ready.


In [6]:
# Cell 5 — source-target leakage sanity test with synthetic example (independent of PISA)
# Demonstrates the identical construction rule on isolated arrays, without changing any PISA data.
from numpy.random import default_rng
rng=default_rng(20261015)
a=rng.integers(1,4,size=(12,32),dtype=np.int8)
b=rng.integers(1,4,size=(12,32),dtype=np.int8)
pm=np.ones((12,32),dtype=bool)
im=np.ones((12,32),dtype=bool)
p=rng.uniform(0,3,size=(12,32,3)).astype(np.float32)
source=np.asarray(ITEM_SET_A)
target=np.asarray(ITEM_SET_B)

def demo_features(cc,bb,perf,inter,px,items):
    c=cc[:,items]; b0=bb[:,items]; per=perf[:,items]; it=inter[:,items]; times=px[:,items,:]
    out=[masked_mean((c.astype(float)-1)/2,per)[0]]
    for j in (1,2,3):out.append(((b0==j)&it).sum(axis=1)/np.maximum(it.sum(axis=1),1))
    for j in range(3):out.append(masked_mean(times[:,:,j],it)[0])
    return np.column_stack(out)

before=demo_features(a,b,pm,im,p,source)
a2=a.copy();b2=b.copy();pm2=pm.copy();im2=im.copy();p2=p.copy()
a2[:,target]=1;b2[:,target]=0;pm2[:,target]=False;im2[:,target]=False;p2[:,target,:]=1e5
after=demo_features(a2,b2,pm2,im2,p2,source)
assert np.array_equal(before,after)
assert not np.array_equal(a[:,target],a2[:,target])
print('Synthetic target-input invariance: PASS (source features do not change when targets change).')

Synthetic target-input invariance: PASS (source features do not change when targets change).


## Identical plausible-value-conditioned outcomes

Copy of Notebook 09's within-education-system least-squares conditioning: for each of the 10 PVs independently, regress **observed hidden-domain mean normalized credit** on [intercept, plausible value] inside each eligible education system, then divide residuals by their within-system sample SD. This is applied separately on the validation and test parts, as in Notebook 09.

**Interpretation warning:** The held-out outcomes are used to define the *analytical target* in each test system. That is not supervised probe fitting, but it means this result is a conditional association analysis, **not** an operational test-time predictor when true item scores are unavailable.

In [7]:
# Cell 6 — exact target conditioning and fixed multitarget Ridge protocol from Notebook 09

def proficiency_conditioned_residual_matrix(domain_score,pv_values,countries,minimum_system_n=30):
    domain_score=np.asarray(domain_score,dtype=float)
    pv_values=np.asarray(pv_values,dtype=float)
    countries=np.asarray(countries)
    assert pv_values.shape==(len(domain_score),10)
    residuals=np.full(pv_values.shape,np.nan,dtype=np.float32)
    for system in np.unique(countries):
        idx=np.flatnonzero(countries==system)
        idx=idx[np.isfinite(domain_score[idx])]
        if len(idx)<minimum_system_n: continue
        y=domain_score[idx].astype(float)
        for j in range(10):
            x=pv_values[idx,j].astype(float)
            valid=np.isfinite(x)&np.isfinite(y)
            if valid.sum()<minimum_system_n: continue
            X=np.column_stack((np.ones(valid.sum()),x[valid]))
            beta,_,_,_=np.linalg.lstsq(X,y[valid],rcond=None)
            resid=y[valid]-X@beta
            sd=resid.std(ddof=1)
            if not np.isfinite(sd) or sd<=0: continue
            residuals[idx[valid],j]=(resid/sd).astype(np.float32)
    return residuals


def fit_multitarget_ridge(X_val,Y_val,X_test):
    valid=np.isfinite(X_val).all(axis=1)&np.isfinite(Y_val).all(axis=1)
    assert valid.sum()>10
    assert np.isfinite(X_test).all()
    scaler=StandardScaler()
    Xv=scaler.fit_transform(X_val[valid]); Xt=scaler.transform(X_test)
    model=Ridge(alpha=ALPHA)
    model.fit(Xv,Y_val[valid])
    return model.predict(Xt)


def safe_rho(y,pred):
    y=np.asarray(y,dtype=float);pred=np.asarray(pred,dtype=float)
    if len(y)<3 or not np.isfinite(y).all() or not np.isfinite(pred).all() or np.std(y)==0 or np.std(pred)==0:
        return np.nan
    return float(spearmanr(y,pred).statistic)

print('Notebook 09-compatible PV conditioning and Ridge(alpha=1) ready.')

Notebook 09-compatible PV conditioning and Ridge(alpha=1) ready.


## Seven-fold / two-direction / four-domain matched evaluation

**CReST control:** `z_general` (64) vs `[z_general, z_domain_specific]` (128).

**Feature baseline:** 36 general features vs `[36 general, 12 source-domain features]` (48).

Predictions for both methods use the *same test row indices, same residual targets and same eligibility mask* for each fold, direction and domain. This notebook uses the saved latent representation arrays from Notebook 09 and does not re-extract latents or load neural checkpoints.

In [8]:
# Cell 7 — locked comparisons. About 56 direction×fold×domain jobs; may take several minutes.
from time import perf_counter
start_time=perf_counter()
fold_rows=[]; system_rows=[]
audit_rows=[]
for fold in range(7):
    validation_fold=(fold+1)%7
    expected_val=np.flatnonzero(fold_values==validation_fold)
    expected_test=np.load(FINAL_TEST_ROOT/f'fold_{fold}'/'test_indices.npy')
    assert np.array_equal(np.sort(expected_test),np.flatnonzero(fold_values==fold))
    assert np.intersect1d(expected_val,expected_test).size==0
    assert len(np.unique(country_values[expected_val]))==9
    assert len(np.unique(country_values[expected_test]))==9

    for direction,(source_items,target_items) in DIRECTIONS.items():
        with np.load(LATENT_DIR/f'fold_{fold}_validation_{direction}.npz') as lat:
            val_idx=np.asarray(lat['handoff_row_index'],dtype=np.int64)
            zg_val=np.asarray(lat['z_general'])
            zd_val=np.asarray(lat['z_domain_specific'])
            da_val=np.asarray(lat['domain_available'],dtype=bool)
        with np.load(LATENT_DIR/f'fold_{fold}_test_{direction}.npz') as lat:
            test_idx=np.asarray(lat['handoff_row_index'],dtype=np.int64)
            zg_test=np.asarray(lat['z_general'])
            zd_test=np.asarray(lat['z_domain_specific'])
            da_test=np.asarray(lat['domain_available'],dtype=bool)
        assert np.array_equal(val_idx,expected_val),f'Validation rows differ from locked fold {fold}'
        assert np.array_equal(test_idx,expected_test),f'Test rows differ from locked fold {fold}'
        assert zg_val.shape==(len(val_idx),64) and zg_test.shape==(len(test_idx),64)
        assert zd_val.shape==(len(val_idx),4,64) and zd_test.shape==(len(test_idx),4,64)
        assert da_val.shape==(len(val_idx),4) and da_test.shape==(len(test_idx),4)
        assert np.isfinite(zg_val).all() and np.isfinite(zg_test).all()
        assert np.isfinite(zd_val).all() and np.isfinite(zd_test).all()

        # General source-only features are shared by all four domain analyses.
        general_val=general_summary(val_idx,source_items)
        general_test=general_summary(test_idx,source_items)

        for domain_id,domain in enumerate(DOMAIN_ORDER):
            source_domain_items=source_items[np.asarray(domain_ids[source_items])==domain_id]
            target_domain_items=target_items[np.asarray(domain_ids[target_items])==domain_id]
            assert len(source_domain_items)>0 and len(target_domain_items)>0
            y_val,n_val=hidden_domain_target(val_idx,target_domain_items)
            y_test,n_test=hidden_domain_target(test_idx,target_domain_items)
            eligible_val=source_domain_observed(val_idx,source_domain_items)&(n_val>=1)&da_val[:,domain_id]
            eligible_test=source_domain_observed(test_idx,source_domain_items)&(n_test>=1)&da_test[:,domain_id]
            assert eligible_val.sum()>=50 and eligible_test.sum()>=50

            Y_val=proficiency_conditioned_residual_matrix(
                y_val[eligible_val],pv_matrix[val_idx[eligible_val]],country_values[val_idx[eligible_val]],MIN_SYSTEM_RESID_N)
            Y_test=proficiency_conditioned_residual_matrix(
                y_test[eligible_test],pv_matrix[test_idx[eligible_test]],country_values[test_idx[eligible_test]],MIN_SYSTEM_RESID_N)
            fit_ok=np.isfinite(Y_val).all(axis=1)
            test_ok=np.isfinite(Y_test).all(axis=1)
            val_eligible=val_idx[eligible_val][fit_ok]
            test_eligible=test_idx[eligible_test][test_ok]
            assert len(val_eligible)>=50 and len(test_eligible)>=50
            Yv=Y_val[fit_ok]
            Yt=Y_test[test_ok]
            # Same eligible rows and outcomes for both methods.
            assert len(Yv)==len(val_eligible) and len(Yt)==len(test_eligible)
            zgv=zg_val[eligible_val][fit_ok]; zgt=zg_test[eligible_test][test_ok]
            zdv=zd_val[eligible_val,domain_id,:][fit_ok]; zdt=zd_test[eligible_test,domain_id,:][test_ok]
            xgv=general_val[eligible_val][fit_ok]; xgt=general_test[eligible_test][test_ok]
            xdv=domain_summary(val_eligible,source_items,domain_id)
            xdt=domain_summary(test_eligible,source_items,domain_id)
            assert xgv.shape[1]==36 and xdv.shape[1]==12

            # The original CReST and feature baseline get the same Yv/Yt.
            pred_cg=fit_multitarget_ridge(zgv,Yv,zgt)
            pred_cc=fit_multitarget_ridge(np.concatenate([zgv,zdv],axis=1),Yv,
                                           np.concatenate([zgt,zdt],axis=1))
            pred_rg=fit_multitarget_ridge(xgv,Yv,xgt)
            pred_rc=fit_multitarget_ridge(np.concatenate([xgv,xdv],axis=1),Yv,
                                           np.concatenate([xgt,xdt],axis=1))

            # Compute system-equal correlations, averaging across 10 PVs within system.
            systems=country_values[test_eligible]
            fold_system=[]
            for system in np.unique(systems):
                m=systems==system
                if m.sum()<MIN_SYSTEM_METRIC_N: continue
                vals=[]
                for j in range(10):
                    vals.append(dict(
                        crest_general_rho=safe_rho(Yt[m,j],pred_cg[m,j]),
                        crest_combined_rho=safe_rho(Yt[m,j],pred_cc[m,j]),
                        ridge_general_rho=safe_rho(Yt[m,j],pred_rg[m,j]),
                        ridge_combined_rho=safe_rho(Yt[m,j],pred_rc[m,j])))
                per_pv=pd.DataFrame(vals)
                assert per_pv.notna().all().all(),f'Undefined Spearman rho: fold={fold}, direction={direction}, domain={domain}, system={system}'
                s=per_pv.mean().to_dict()
                s['crest_delta_rho']=s['crest_combined_rho']-s['crest_general_rho']
                s['ridge_delta_rho']=s['ridge_combined_rho']-s['ridge_general_rho']
                s['delta_difference']=s['crest_delta_rho']-s['ridge_delta_rho']
                s['combined_difference']=s['crest_combined_rho']-s['ridge_combined_rho']
                row={'outer_fold':fold,'direction':direction,'domain':domain,'country':system,
                     'n_students':int(m.sum()),**s}
                system_rows.append(row);fold_system.append(row)

            # For detailed diagnostics also keep fold-pooled PV-average metrics.
            pv_means=[]
            for j in range(10):
                pv_means.append({'crest_general_rho':safe_rho(Yt[:,j],pred_cg[:,j]),
                                 'crest_combined_rho':safe_rho(Yt[:,j],pred_cc[:,j]),
                                 'ridge_general_rho':safe_rho(Yt[:,j],pred_rg[:,j]),
                                 'ridge_combined_rho':safe_rho(Yt[:,j],pred_rc[:,j])})
            pooled=pd.DataFrame(pv_means).mean().to_dict()
            system_means=pd.DataFrame(fold_system)
            fold_rows.append({'outer_fold':fold,'direction':direction,'domain':domain,
                              'n_source_domain_items':len(source_domain_items),
                              'n_target_domain_items':len(target_domain_items),
                              'n_validation_students':int(len(val_eligible)),
                              'n_test_students':int(len(test_eligible)),
                              'n_metric_systems':int(len(fold_system)),
                              **{'pooled_'+k:v for k,v in pooled.items()},
                              **{'system_equal_'+k:float(system_means[k].mean())
                                 for k in ['crest_general_rho','crest_combined_rho','ridge_general_rho','ridge_combined_rho','crest_delta_rho','ridge_delta_rho','delta_difference','combined_difference']}})
            audit_rows.append({'outer_fold':fold,'direction':direction,'domain':domain,
                               'n_val_eligible':int(eligible_val.sum()),'n_test_eligible':int(eligible_test.sum()),
                               'n_val_complete_pv':int(len(val_eligible)),'n_test_complete_pv':int(len(test_eligible)),
                               'source_items':','.join(map(str,source_domain_items)),
                               'target_items':','.join(map(str,target_domain_items))})
            print(f'fold={fold} {direction:6s} {domain:10s} | '+
                  f'CReST Δρ={system_means["crest_delta_rho"].mean():+.4f}, '+
                  f'Ridge Δρ={system_means["ridge_delta_rho"].mean():+.4f}, '+
                  f'CReST−Ridge ΔΔρ={system_means["delta_difference"].mean():+.4f}',flush=True)

    print(f'Fold {fold} completed. Elapsed {(perf_counter()-start_time)/60:.1f} min',flush=True)

system_df=pd.DataFrame(system_rows)
fold_df=pd.DataFrame(fold_rows)
audit_df=pd.DataFrame(audit_rows)
assert len(fold_df)==56 and len(audit_df)==56
assert system_df.groupby(['country','direction','domain']).size().eq(1).all()
fold_df.to_csv(OUTPUT_DIR/'n23_fold_direction_domain_results.csv',index=False)
system_df.to_csv(OUTPUT_DIR/'n23_system_direction_domain_results.csv',index=False)
audit_df.to_csv(OUTPUT_DIR/'n23_eligibility_and_items.csv',index=False)
print(f'Completed {len(fold_df)} comparisons; {len(system_df)} system×direction×domain rows.')

fold=0 A_to_B Written    | CReST Δρ=+0.0251, Ridge Δρ=+0.0221, CReST−Ridge ΔΔρ=+0.0030
fold=0 A_to_B Visual     | CReST Δρ=+0.0065, Ridge Δρ=+0.0263, CReST−Ridge ΔΔρ=-0.0199
fold=0 A_to_B Social     | CReST Δρ=+0.2471, Ridge Δρ=+0.0655, CReST−Ridge ΔΔρ=+0.1816
fold=0 A_to_B Scientific | CReST Δρ=+0.1383, Ridge Δρ=+0.0107, CReST−Ridge ΔΔρ=+0.1276
fold=0 B_to_A Written    | CReST Δρ=+0.0898, Ridge Δρ=+0.0290, CReST−Ridge ΔΔρ=+0.0608
fold=0 B_to_A Visual     | CReST Δρ=-0.0123, Ridge Δρ=+0.0207, CReST−Ridge ΔΔρ=-0.0329
fold=0 B_to_A Social     | CReST Δρ=+0.1410, Ridge Δρ=+0.0645, CReST−Ridge ΔΔρ=+0.0764
fold=0 B_to_A Scientific | CReST Δρ=+0.0624, Ridge Δρ=-0.0022, CReST−Ridge ΔΔρ=+0.0646
Fold 0 completed. Elapsed 0.1 min
fold=1 A_to_B Written    | CReST Δρ=+0.0695, Ridge Δρ=+0.0188, CReST−Ridge ΔΔρ=+0.0507
fold=1 A_to_B Visual     | CReST Δρ=+0.0343, Ridge Δρ=+0.0201, CReST−Ridge ΔΔρ=+0.0142
fold=1 A_to_B Social     | CReST Δρ=+0.2323, Ridge Δρ=+0.0951, CReST−Ridge ΔΔρ=+0.1372
fold=1 A_

## Reproduce original CReST results before comparing

The saved CReST domain increments from Notebook 09 provide a **strong integrity check**. On the same system/direction/domain and same eligible students, the CReST baseline reproduced here should match the original `crest_profile_beyond_pisa_ct_system.csv` within numerical tolerance. A mismatch means **do not interpret the comparative results** until the source of the discrepancy is resolved.

In [9]:
# Cell 8 — strict original-CReST reproducibility audit
original=pd.read_csv(PROFILE_DIR/'crest_profile_beyond_pisa_ct_system.csv')
required_original={'outer_fold','direction','domain','country','n_students','general_rho','combined_rho','delta_rho'}
assert required_original.issubset(original.columns), 'Unexpected Notebook 09 results schema'
assert len(original)==len(system_df),f'CReST system coverage differs: old={len(original)}, new={len(system_df)}'
keys=['outer_fold','direction','domain','country']
merged=system_df.merge(original[keys+['n_students','general_rho','combined_rho','delta_rho']],on=keys,how='outer',indicator=True,validate='one_to_one',suffixes=('_n23','_n09'))
assert merged['_merge'].eq('both').all(), 'CReST system coverage mismatch'
assert merged['n_students_n23'].eq(merged['n_students_n09']).all(), 'Eligibility/student counts differ from Notebook 09'
check={'crest_general_rho':'general_rho','crest_combined_rho':'combined_rho','crest_delta_rho':'delta_rho'}
for newcol,oldcol in check.items():
    err=np.abs(merged[f'{newcol}']-merged[f'{oldcol}'])
    print(f'{newcol:23s} vs Notebook09: max_abs_difference={err.max():.8f}')
    assert np.isfinite(err).all() and float(err.max())<1e-4, (
        f'Notebook 09 mismatch for {newcol}; verify unchanged latent archives/PV cohort/probe fitting.')
print('Original CReST domain profile reproduction: PASS.')

crest_general_rho       vs Notebook09: max_abs_difference=0.00000000
crest_combined_rho      vs Notebook09: max_abs_difference=0.00000000
crest_delta_rho         vs Notebook09: max_abs_difference=0.00000000
Original CReST domain profile reproduction: PASS.


## Paired cross-system effect sizes and 95% bootstrap CIs

For each domain, first average the **two directions within each education system**, then average the **63 systems equally**. Bootstrap resamples entire systems with replacement (10,000 draws), keeping all paired model contrasts in the same system. Report both:

- **Combined-model accuracy difference:** `CReST general+domain ρ − feature Ridge general+domain ρ`
- **Incremental-information difference (ΔΔρ):** `(CReST combined − CReST general) − (Ridge combined − Ridge general)`

A large Δρ alone is not necessarily better performance if the general baseline is poor; **check combined-model correlation as well**. Confidence intervals describe uncertainty over the observed systems, but do not undo earlier model selection, shared PISA assessment, or this post-hoc comparison.

In [10]:
# Cell 9 — 63-system paired bootstrap (when the source data support all systems)
SYSTEM_METRICS=['crest_general_rho','crest_combined_rho','ridge_general_rho','ridge_combined_rho',
                'crest_delta_rho','ridge_delta_rho','delta_difference','combined_difference']
system_domain=(system_df.groupby(['country','domain'],as_index=False)
               .agg(**{m:(m,'mean') for m in SYSTEM_METRICS},
                    n_directions=('direction','nunique')))
assert system_domain['n_directions'].eq(2).all(), 'At least one country-domain has a missing transfer direction'
boot_rng=np.random.default_rng(BOOTSTRAP_SEED)
summary_rows=[]
for domain in DOMAIN_ORDER:
    frame=system_domain.loc[system_domain['domain']==domain].sort_values('country')
    values=frame[SYSTEM_METRICS].to_numpy(dtype=float)
    assert len(frame)==63, f'{domain}: expected 63 systems, found {len(frame)}'
    assert np.isfinite(values).all()
    sample_ids=boot_rng.integers(0,len(frame),size=(BOOTSTRAP_REPS,len(frame)))
    # Small (10k × 63 × 8), 63-system paired bootstrap.
    boot=values[sample_ids].mean(axis=1)
    means=values.mean(axis=0)
    row={'domain':domain,'n_systems':len(frame),'bootstrap_reps':BOOTSTRAP_REPS}
    for j,m in enumerate(SYSTEM_METRICS):
        lo,hi=np.percentile(boot[:,j],[2.5,97.5])
        row[m]=float(means[j]);row[m+'_ci_low']=float(lo);row[m+'_ci_high']=float(hi)
    row['n_systems_crest_delta_gt_ridge_delta']=int((frame['delta_difference']>0).sum())
    row['n_systems_crest_combined_gt_ridge_combined']=int((frame['combined_difference']>0).sum())
    summary_rows.append(row)
summary=pd.DataFrame(summary_rows)
system_domain.to_csv(OUTPUT_DIR/'n23_63system_paired_domain_results.csv',index=False)
summary.to_csv(OUTPUT_DIR/'n23_main_comparison_with_bootstrap.csv',index=False)
cols=['domain','crest_general_rho','crest_combined_rho','crest_delta_rho','ridge_general_rho','ridge_combined_rho','ridge_delta_rho','delta_difference','combined_difference','delta_difference_ci_low','delta_difference_ci_high']
print(summary[cols].round(4).to_string(index=False))

    domain  crest_general_rho  crest_combined_rho  crest_delta_rho  ridge_general_rho  ridge_combined_rho  ridge_delta_rho  delta_difference  combined_difference  delta_difference_ci_low  delta_difference_ci_high
   Written             0.1274              0.1945           0.0671             0.1176              0.1403           0.0227            0.0444               0.0542                   0.0358                    0.0531
    Visual             0.0560              0.0748           0.0188             0.0648              0.0893           0.0245           -0.0057              -0.0145                  -0.0196                    0.0082
    Social             0.1581              0.3304           0.1723             0.2040              0.2743           0.0703            0.1021               0.0562                   0.0918                    0.1123
Scientific             0.1224              0.2084           0.0860             0.1462              0.1476           0.0014            0.0846        

In [11]:
# Cell 10 — lightweight publication-ready table and unambiguous interpretation boundaries
paper=summary[['domain','n_systems','crest_general_rho','crest_combined_rho','crest_delta_rho',
               'ridge_general_rho','ridge_combined_rho','ridge_delta_rho',
               'combined_difference','delta_difference',
               'combined_difference_ci_low','combined_difference_ci_high',
               'delta_difference_ci_low','delta_difference_ci_high']].copy()
paper.to_csv(OUTPUT_DIR/'n23_manuscript_ready_comparison.csv',index=False)
print('Publication table (system-equal averages over A/B and PV1–PV10):')
print(paper[['domain','crest_combined_rho','ridge_combined_rho','crest_delta_rho','ridge_delta_rho','delta_difference']].round(4).to_string(index=False))

protocol={
  'analysis_name':'CReST-vs-source-summary Ridge: PISA Creative Thinking domain performance conditional on official proficiency',
  'analysis_status':'exploratory_post_hoc',
  'crest_version':'original_published_CReST_from_notebook09',
  'v2_included':False,
  'supervised_probe':'StandardScaler + Ridge(alpha=1.0), fitted on validation systems only',
  'outer_folds':7,'countries':63,'directions':list(DIRECTIONS),
  'item_sets':{'A':ITEM_SET_A.tolist(),'B':ITEM_SET_B.tolist()},
  'domain_order':DOMAIN_ORDER,
  'features':{'feature_ridge_general':36,'feature_ridge_domain_added':12,
              'crest_general_latent':64,'crest_domain_added':64},
  'pv_columns':PV_COLUMNS,'pvs_never_model_inputs':True,
  'conditional_target':'within-system OLS domain_score~[intercept,PV] separately per PV; residual / sample SD; identical to Notebook09',
  'minimum_system_residual_n':MIN_SYSTEM_RESID_N,'minimum_metric_n':MIN_SYSTEM_METRIC_N,
  'uncertainty':'paired 63-system percentile bootstrap, 10000 resamples',
  'limitations':['Scored-item A/B split does not hold out whole tasks.',
                 'PV-conditioned target calculated separately within validation/test system using outcome data.',
                 'Different input dimensions and manually engineered feature availability.',
                 'Post-hoc comparison on an assessment already used for model development.',
                 'No independent construct validation, no causality, no individual diagnostic use.'],
}
with open(OUTPUT_DIR/'n23_methodological_protocol.json','w',encoding='utf-8') as f:
    json.dump(protocol,f,indent=2,ensure_ascii=False)
print('Protocol saved. No manuscript claims should be updated without reviewing these actual results.')

Publication table (system-equal averages over A/B and PV1–PV10):
    domain  crest_combined_rho  ridge_combined_rho  crest_delta_rho  ridge_delta_rho  delta_difference
   Written              0.1945              0.1403           0.0671           0.0227            0.0444
    Visual              0.0748              0.0893           0.0188           0.0245           -0.0057
    Social              0.3304              0.2743           0.1723           0.0703            0.1021
Scientific              0.2084              0.1476           0.0860           0.0014            0.0846
Protocol saved. No manuscript claims should be updated without reviewing these actual results.


In [12]:
# Cell 11 — ZIP outputs only, exclude individual-level data or original OECD records
paths=sorted(OUTPUT_DIR.glob('n23_*.csv')) + sorted(OUTPUT_DIR.glob('n23_*.json'))
assert len(paths)>=7, f'Expected at least 7 output files, got {len(paths)}'
zip_path=BASE_DIR/'crest_vs_ridge_domain_relative_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in paths:z.write(p,arcname=p.name)
with zipfile.ZipFile(zip_path) as z:
    assert len(z.namelist())==len(paths)
print('Finished. ZIP:',zip_path)
print('Exported files:',*[p.name for p in paths],sep='\n  ')
print('Student-level PISA records were not included in ZIP.')

Finished. ZIP: D:\Kubra (Umit blg)\Kubra (papers)\PISA-CReST\Dataset and codes\crest_vs_ridge_domain_relative_results.zip
Exported files:
  n23_63system_paired_domain_results.csv
  n23_eligibility_and_items.csv
  n23_fold_direction_domain_results.csv
  n23_main_comparison_with_bootstrap.csv
  n23_manuscript_ready_comparison.csv
  n23_system_direction_domain_results.csv
  n23_methodological_protocol.json
Student-level PISA records were not included in ZIP.


### How to interpret

The output answers three different questions: (1) which method has higher **absolute domain prediction correlation** after conditioning on official proficiency; (2) which method gains more from adding domain information beyond its general representation (**Δρ**); and (3) whether differences are consistent across education systems. The **paired ΔΔρ** is more informative than comparing Δρ columns informally, but the absolute combined-model correlations must always be shown alongside.

**Not a fair architecture superiority test:** CReST uses learned representations, while the Ridge baseline uses direct hand-crafted aggregates; the models have different input feature counts and the source task exposure structure can remain predictive. It cannot establish that Transformers are needed, that learner abilities are separate constructs, or that domain performance transfers beyond PISA tasks. Regardless of whether CReST or Ridge performs better, include the results and limitations transparently.

**Runtime:** This is a CPU-only set of Ridge fittings, **no 4-epoch CReST training**. It may still take several minutes because all 7 folds × 2 directions × 4 domains × 10 plausible values are evaluated. Progress prints after every domain and fold; there are no long silent neural-network training epochs.